# 04 · Single book, end to end

Run the **production pipeline** (built by the factory from `pipeline.toml`) on
one cached book and inspect every artifact: both PDFs, the OCR sidecar, the
cover, bookmarks and the PDF/A verdict.

In [ ]:
import sys
from pathlib import Path

# project importable when the kernel isn't the project .venv
sys.path.insert(0, str(Path.cwd().parent))

from evilflowers_books_digitalizer.runtime import load_runtime
from evilflowers_books_digitalizer.cache import LocalCache

# ONE source of truth for cache_dir + output_dir (paths resolve to the project
# root regardless of the notebook's working directory) — this is the unified
# cache every notebook and the production runner share.
rt = load_runtime()
cache = LocalCache(rt.cache_dir)
print("cache :", rt.cache_dir)
print("output:", rt.output_dir)

In [ ]:
import pymupdf

def show_pdf_page(pdf_path, page=1, dpi=110):
    """Render one PDF page inline (for visual QA)."""
    from IPython.display import Image, display
    d = pymupdf.open(str(pdf_path))
    page = min(page, len(d) - 1)
    display(Image(data=d[page].get_pixmap(dpi=dpi).tobytes("png")))

def page_render_ms(pdf_path, dpi=150):
    """Mean per-page render time (ms) — the viewer-load-speed proxy."""
    import time
    d = pymupdf.open(str(pdf_path))
    ts = []
    for i in range(len(d)):
        t = time.time(); d[i].get_pixmap(dpi=dpi); ts.append((time.time() - t) * 1000)
    return sum(ts) / len(ts)

def cached_books():
    """(source, book_id) for every book staged in the local cache."""
    scans = rt.cache_dir / "scans"
    return sorted((p.parent.name, p.name) for p in scans.glob("*/*") if p.is_dir())

In [ ]:
from evilflowers_books_digitalizer import build_source
from evilflowers_books_digitalizer.pipeline.base import BookContext
from evilflowers_books_digitalizer.pipeline.factory import build_pipeline
from evilflowers_books_digitalizer.pipeline.layout import book_dir
from evilflowers_books_digitalizer.runtime import build_catalog

src, bid = cached_books()[0]
source = build_source(rt.source, src)
catalog = build_catalog(rt.metadata)
pipeline = build_pipeline(source, cache, config=rt.config, catalog=catalog)
print(pipeline)

In [ ]:
# output_dir must match the configured [render] layout (flat | split)
layout = rt.config.get("render", {}).get("layout", "flat")
ctx = BookContext(source=src, book_id=bid, work_dir=cache.book_dir(src, bid),
                  output_dir=book_dir(rt.output_dir, src, layout))
ctx = pipeline.run(ctx)
for name, info in ctx.metadata.get("outputs", {}).items():
    print(f"{name:13} {info['mb']:6.2f} MB  {info['path']}")
print("cover:", ctx.artifacts.get("cover"))
print("bookmarks:", ctx.metadata.get("n_bookmarks"), "| pdfa:", ctx.metadata.get("pdfa_valid"))

### Distribution copy — first content page

In [ ]:
show_pdf_page(ctx.artifacts['pdf_distribution'], page=2)